# 🧠 EEG Preprocessing Pipeline — Stage 2: Average Reference + ICA Fit

**Dataset:** `ds006780` (ASD EEG, BIDS-formatted)  
**Input:** `desc-preproc_eeg.fif` files from Stage 1 (filtered, resampled, bads interpolated)  
**Output:** `desc-preproc_ica.fif` files (ICA decomposition only — no application yet)

---

## What this stage does

Applies average reference to each recording, then fits an ICA decomposition and saves the  
**decomposition object** (mixing/unmixing matrices + metadata, ~MB per file).  
Component **labeling** and **application** happen in Stage 3.

```
desc-preproc_eeg.fif  →  average reference  →  pick EEG channels  →  fit ICA  →  save  →  desc-preproc_ica.fif
```

## Why re-reference BEFORE ICA?

Average reference is applied here — not after ICA — following MNE best practice (Delorme et al.):

1. **Neurophysiological correctness:** Average reference distributes the scalp field symmetrically,
   so ICA components map more cleanly to real brain and artifact sources.
2. **Rank is handled automatically:** Average reference removes one degree of freedom (rank − 1).
   Setting `ICA_N_COMPONENTS = None` tells MNE to detect rank automatically — it will fit
   63 components on 64-channel data, which is exactly correct.

## Why split fit / label / apply?

ICA fitting is the slow, deterministic part. Once fit, you can revisit labeling and rejection  
criteria without re-running the expensive decomposition. This is the standard pattern in  
research pipelines (mne-bids-pipeline, PyLossless, EEGLAB-AMICA).

## Alignment with Stage 1

| Concept | Stage 1 | Stage 2 |
|---|---|---|
| Pipeline name | `mne-preproc-pre-ica` | `mne-ica-fit` |
| Output entity | `desc-preproc_eeg.fif` | `desc-preproc_ica.fif` |
| Folder structure | `derivatives/<pipeline>/sub-XXX/eeg/` | same |
| Logging | CSV per stage | CSV per stage |
| Resume logic | skip if output exists | skip if output exists |
| Smoke test | yes | yes |


---
# 1. Setup

## 1.1 Install dependencies (run once)

`python-picard` is the recommended ICA algorithm (faster + more reliable than FastICA/InfoMax).

In [1]:
 !pip install mne mne-bids pandas numpy joblib tqdm python-picard


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


## 1.2 Imports

In [2]:
import gc
import json
import time
import traceback
from pathlib import Path
from datetime import datetime
from contextlib import contextmanager

import numpy as np
import pandas as pd
import mne
from mne.preprocessing import ICA
from joblib import Parallel, delayed
from tqdm.auto import tqdm

# GCS is only needed when STAGE1_SOURCE == 'gcs' or UPLOAD_STAGE2_TO_GCS == True
try:
    from google.cloud import storage as gcs_lib
    from google.cloud.storage.retry import DEFAULT_RETRY
    HAS_GCS = True
except ImportError:
    HAS_GCS = False

# Picard is optional — fall back to InfoMax if not installed
try:
    import picard  # noqa: F401
    HAS_PICARD = True
except ImportError:
    HAS_PICARD = False

mne.set_log_level('WARNING')
print(f'MNE version:       {mne.__version__}')
print(f'Picard available:  {HAS_PICARD}')
print(f'GCS available:     {HAS_GCS}')


MNE version:       1.10.2
Picard available:  True
GCS available:     True


---
# 2. Configuration

## 2.1 Paths

**Input** comes from Stage 1's derivatives folder.  
**Output** goes into a new BIDS-derivative dataset (`mne-ica-fit`).

In [3]:
# ─── Stage 1 OUTPUT becomes Stage 2 INPUT ─────────────────────────────────────
PROJECT_ROOT     = Path.home() / 'asd_eeg_pipeline'
STAGE1_PIPELINE  = 'mne-preproc-pre-ica'
STAGE1_DERIV     = PROJECT_ROOT / 'derivatives' / STAGE1_PIPELINE

# ─── Stage 2 OUTPUT (new BIDS-derivative dataset) ─────────────────────────────
DATASET_ID       = 'ds006780'
PIPELINE_NAME    = 'mne-ica-fit'
DERIV_ROOT       = PROJECT_ROOT / 'derivatives' / PIPELINE_NAME
LOG_DIR          = PROJECT_ROOT / 'derivatives' / 'logs'

for d in (DERIV_ROOT, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'📥 Input  (Stage 1):  {STAGE1_DERIV}  (exists: {STAGE1_DERIV.exists()})')
print(f'📤 Output (Stage 2):  {DERIV_ROOT}')

📥 Input  (Stage 1):  /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-preproc-pre-ica  (exists: True)
📤 Output (Stage 2):  /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-fit


In [4]:
# ─── Stage 1 source: local disk OR GCS ────────────────────────────────────────
# 'local' = read directly from STAGE1_DERIV (on-disk derivatives from Stage 1)
# 'gcs'   = stream from gs://{GCS_BUCKET}/{GCS_STAGE1_PREFIX}/, cache in /tmp
STAGE1_SOURCE = 'local'   # ← changed from 'gcs'

GCS_BUCKET           = 'asd-eeg-dataset'
GCS_STAGE1_PREFIX    = f'derivatives/{DATASET_ID}/{STAGE1_PIPELINE}'
GCS_STAGE2_PREFIX    = f'derivatives/{DATASET_ID}/{PIPELINE_NAME}'

# Cache for streamed Stage 1 files (only used if STAGE1_SOURCE == 'gcs')
STAGE1_CACHE = Path('/tmp/stage1_cache')
STAGE1_CACHE.mkdir(parents=True, exist_ok=True)

# Set True if you want to back up Stage 2 ICA outputs to GCS after fitting
UPLOAD_STAGE2_TO_GCS = False   # ← changed from True


In [5]:
# Only initialise GCS client if we actually need it
gcs_client = None
gcs_bucket = None
GCS_RETRY   = None
GCS_TIMEOUT = 600
GCS_CHUNK   = 16 * 1024 * 1024

if STAGE1_SOURCE == 'gcs' or UPLOAD_STAGE2_TO_GCS:
    if not HAS_GCS:
        raise RuntimeError(
            'google-cloud-storage is not installed. '
            'Run: pip install google-cloud-storage'
        )
    gcs_client  = gcs_lib.Client()
    gcs_bucket  = gcs_client.bucket(GCS_BUCKET)
    GCS_RETRY   = DEFAULT_RETRY.with_deadline(600).with_delay(
                      initial=2, maximum=30, multiplier=2)
    print(f'✅ GCS client ready  →  gs://{GCS_BUCKET}/')
else:
    print('ℹ️  GCS client skipped (STAGE1_SOURCE=local, UPLOAD_STAGE2_TO_GCS=False)')


ℹ️  GCS client skipped (STAGE1_SOURCE=local, UPLOAD_STAGE2_TO_GCS=False)


## 2.2 ICA parameters

**Method**: Picard with `extended=True, ortho=False` is mathematically equivalent to extended 
InfoMax but ~5–10× faster. This is what mne-bids-pipeline uses by default.

**n_components**: 
- `None` → MNE picks based on data rank (recommended after bad-channel interpolation, since 
  interpolated channels reduce rank)
- `0.99` → enough components to explain 99% variance  
- An integer → fixed count (only safe if rank is known and consistent across files)

**decim**: take every Nth sample during fitting. Speeds things up 2–4× with negligible quality 
loss. For 250 Hz data, `decim=2` is fine; for 500 Hz, `decim=3` works well.

In [6]:
try:
    import picard
    HAS_PICARD = True
except ImportError:
    HAS_PICARD = False
    
# ─── ICA algorithm ────────────────────────────────────────────────────────────
ICA_METHOD       = 'picard' if HAS_PICARD else 'infomax'
ICA_FIT_PARAMS   = (dict(ortho=False, extended=True)   # Picard → extended-InfoMax behavior
                    if HAS_PICARD else
                    dict(extended=True))               # plain extended InfoMax fallback

# ─── Decomposition shape ──────────────────────────────────────────────────────
ICA_N_COMPONENTS = None    # None = use rank (recommended); or float (variance) / int (fixed)
ICA_MAX_ITER     = 500     # Picard usually converges in <200; allow headroom
ICA_RANDOM_STATE = 42      # reproducibility — KEEP this fixed across runs

# ─── Performance ──────────────────────────────────────────────────────────────
ICA_DECIM        = 3       # subsample during fit (None = use all samples)

# ─── Channel selection ────────────────────────────────────────────────────────
ICA_PICKS        = 'eeg'   # fit on EEG only — EOG/ECG used later for component labeling

## 2.3 Tasks and execution

In [7]:
# Process every task by default; restrict here if you want to do only a subset
TASKS        = ['Restingstate', 'FAST', 'IC', 'motor']   # or None for all found

N_JOBS_INNER = 4       # MNE-internal threading (limited use during ICA fit)
N_JOBS_OUTER = 1       # parallel files. Bump to 4-8 ONLY if you have RAM headroom —
                       # ICA fit memory peaks at ~3-5x raw size.
OVERWRITE    = False   # if True, refit files that already have an ICA output

---
# 3. Initialize BIDS-derivatives dataset

Same pattern as Stage 1 — proper `dataset_description.json` so this output is itself a 
valid BIDS-derivative dataset that points to its source (Stage 1's output).

In [8]:
def init_bids_derivatives(deriv_root, pipeline_name, source_pipeline_name):
    """Write dataset_description.json for the Stage 2 derivative."""
    desc = {
        'Name': f'{DATASET_ID} — {pipeline_name}',
        'BIDSVersion': '1.9.0',
        'DatasetType': 'derivative',
        'GeneratedBy': [{
            'Name': pipeline_name,
            'Version': '0.1.0',
            'Description': (
                f'ICA decomposition fit on Stage 1 output. '
                f'Method: {ICA_METHOD} ({ICA_FIT_PARAMS}). '
                f'Decomposition only — components are not yet labeled or applied.'
            ),
            'CodeURL': 'local notebook',
        }],
        'SourceDatasets': [{
            'DOI': 'n/a',
            'URL': f'derivatives/{source_pipeline_name}',
        }],
    }
    out = Path(deriv_root) / 'dataset_description.json'
    out.write_text(json.dumps(desc, indent=2))
    return out

desc_path = init_bids_derivatives(DERIV_ROOT, PIPELINE_NAME, STAGE1_PIPELINE)
print(f'✅ Wrote {desc_path}')

✅ Wrote /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-fit/dataset_description.json


---
# 4. Core pipeline functions

Same modular pattern as Stage 1 — small single-purpose functions, with `process_one` 
as the orchestrator.

## 4.1 Discovery — find Stage 1 output files

In [9]:
def parse_bids_filename(fname):
    """Pull subject/task/run out of a BIDS filename.

    Example: 'sub-10003_task-Restingstate_run-01_desc-preproc_eeg.fif'
        → {'subject': '10003', 'task': 'Restingstate', 'run': '01'}
    """
    parts = Path(fname).stem.split('_')
    fields = {}
    for p in parts:
        if '-' in p:
            key, val = p.split('-', 1)
            fields[key] = val
    return {
        'subject': fields.get('sub'),
        'task':    fields.get('task'),
        'run':     fields.get('run'),
        'desc':    fields.get('desc'),
    }


def discover_preprocessed_files(stage1_root, tasks=None):
    """Find every Stage 1 output file. Returns list of (path, entities) tuples."""
    stage1_root = Path(stage1_root)
    files = sorted(stage1_root.rglob('*_desc-preproc_eeg.fif'))
    out = []
    for f in files:
        ent = parse_bids_filename(f.name)
        if tasks is not None and ent['task'] not in tasks:
            continue
        out.append((f, ent))
    return out

In [10]:
def discover_preprocessed_files_from_gcs(prefix, tasks=None):
    """List Stage 1 outputs in GCS. Returns list of (gcs_blob_name, entities)."""
    out = []
    for blob in gcs_client.list_blobs(GCS_BUCKET, prefix=f'{prefix}/'):
        if not blob.name.endswith('_desc-preproc_eeg.fif'):
            continue
        ent = parse_bids_filename(Path(blob.name).name)
        if tasks is not None and ent['task'] not in tasks:
            continue
        out.append((blob.name, ent))
    return out


@contextmanager
def staged_from_gcs(blob_name, cache_dir=STAGE1_CACHE):
    """Download one Stage 1 .fif from GCS into local cache, yield path, delete after."""
    local_path = cache_dir / Path(blob_name).name
    if not local_path.exists():
        blob = gcs_bucket.blob(blob_name, chunk_size=GCS_CHUNK)
        blob.download_to_filename(str(local_path), timeout=GCS_TIMEOUT, retry=GCS_RETRY)
    try:
        yield local_path
    finally:
        try:
            local_path.unlink(missing_ok=True)
        except OSError:
            pass


def upload_stage2_to_gcs(local_path: Path) -> str:
    """Upload Stage 2 ICA output to GCS."""
    rel = local_path.relative_to(DERIV_ROOT)
    blob_name = f'{GCS_STAGE2_PREFIX}/{rel.as_posix()}'
    blob = gcs_bucket.blob(blob_name, chunk_size=GCS_CHUNK)
    if blob.exists():
        blob.reload()
        if blob.size == local_path.stat().st_size:
            return f'gs://{GCS_BUCKET}/{blob_name}'
    blob.upload_from_filename(str(local_path), timeout=GCS_TIMEOUT, retry=GCS_RETRY)
    return f'gs://{GCS_BUCKET}/{blob_name}'

## 4.2 Loading

In [11]:
def load_preprocessed(fpath, preload=True):
    """Load a Stage 1 preprocessed .fif file."""
    raw = mne.io.read_raw_fif(fpath, preload=preload, verbose=False)
    return raw

## 4.2b Average Reference

Applies average reference **before** ICA. See the notebook header for the full rationale.

- `projection=True` adds the reference as an MNE projector rather than modifying data in place.
- `apply_proj()` bakes the projector in immediately so ICA sees the re-referenced signal.
- Only channels of type `EEG` are affected; MISC/EXG channels are untouched.
- The `rank_reduction=1` is logged per file — lets you confirm `n_components_fit = n_eeg_channels − 1`.


In [12]:
def apply_average_reference(raw):
    raw.set_eeg_reference('average', projection=True, verbose=False)
    raw.apply_proj(verbose=False)
    return raw, 1


## 4.3 ICA fit

Picks only EEG channels (EOG/ECG are kept aside for component labeling in Stage 3).  
Times the fit and reports convergence stats so you can audit any pathological cases.

In [13]:
def fit_ica(raw,
            method=ICA_METHOD,
            n_components=ICA_N_COMPONENTS,
            max_iter=ICA_MAX_ITER,
            random_state=ICA_RANDOM_STATE,
            fit_params=None,
            decim=ICA_DECIM,
            picks=ICA_PICKS):
    """Fit ICA on a Raw object. Returns (ica, fit_info_dict)."""
    if fit_params is None:
        fit_params = ICA_FIT_PARAMS

    ica = ICA(
        n_components=n_components,
        method=method,
        fit_params=fit_params,
        max_iter=max_iter,
        random_state=random_state,
    )

    t0 = time.time()
    ica.fit(raw, picks=picks, decim=decim, verbose=False)
    fit_seconds = time.time() - t0

    fit_info = {
        'n_components_fit': ica.n_components_,
        'n_iter':           getattr(ica, 'n_iter_', None),
        'fit_seconds':      round(fit_seconds, 1),
        'method':           method,
        'decim':            decim,
        'converged':        ica.n_iter_ < max_iter if hasattr(ica, 'n_iter_') else None,
    }
    return ica, fit_info

## 4.4 Saving

MNE convention: ICA files end in `_ica.fif` so MNE recognizes them on load. We combine 
that with the BIDS `desc-` entity → `..._desc-preproc_ica.fif`.

In [14]:
def ica_output_path(entities, deriv_root, desc_label='preproc'):
    """Build BIDS-style output path for the ICA decomposition.

    Output:  <deriv_root>/sub-XXX/eeg/sub-XXX_task-Y_run-ZZ_desc-<label>_ica.fif

    The `_ica.fif` suffix is what MNE expects for ICA decomposition files.
    """
    sub = entities['subject']
    out_dir = Path(deriv_root) / f'sub-{sub}' / 'eeg'
    out_dir.mkdir(parents=True, exist_ok=True)

    parts = [f'sub-{sub}', f'task-{entities["task"]}']
    if entities.get('run'):
        parts.append(f'run-{entities["run"]}')
    parts.extend([f'desc-{desc_label}', 'ica'])
    return out_dir / ('_'.join(parts) + '.fif')


def save_ica(ica, entities, deriv_root, desc_label='preproc'):
    """Save the ICA decomposition to BIDS-derivative path."""
    out = ica_output_path(entities, deriv_root, desc_label)
    ica.save(out, overwrite=True, verbose=False)
    return out

## 4.5 Logging utility

In [15]:
def append_log(log_path, record):
    """Append one row to a CSV log, creating with headers if needed.

    Streams to disk so we don't accumulate state in RAM across thousands of files.
    """
    record = {**record, 'timestamp': datetime.now().isoformat(timespec='seconds')}
    df = pd.DataFrame([record])
    header = not Path(log_path).exists()
    df.to_csv(log_path, mode='a', header=header, index=False)

## 4.6 The orchestrator — `process_one`

Per-file pipeline: load → fit → save. **Resumable**, **logged**, **memory-safe**.

The `finally` block is critical: ICA fit allocates whitening matrices that can peak at 
3–5× raw size. We must release them before the next file.

In [16]:
LOG_FILE  = LOG_DIR / '02_ica_fit_log.csv'
ERROR_LOG = LOG_DIR / '02_ica_fit_errors.txt'


def process_one(item, deriv_root=DERIV_ROOT, log_file=LOG_FILE):
    """
    Per-file pipeline: load → average reference → fit ICA → save.

    item: (source_ref, entities) where source_ref is either:
      - a Path to a local .fif (when STAGE1_SOURCE == 'local')
      - a GCS blob name       (when STAGE1_SOURCE == 'gcs')
    """
    source_ref, entities = item
    base = {k: entities.get(k) for k in ('subject', 'task', 'run')}
    out_path = ica_output_path(entities, deriv_root, desc_label='preproc')

    if not OVERWRITE and out_path.exists():
        result = {**base, 'status': 'skipped', 'out': str(out_path)}
        if log_file is not None:   # ✅ اینو اضافه کن
            append_log(log_file, result)
        return result

    raw = None
    try:
        # Get a local file path either way
        if STAGE1_SOURCE == 'gcs':
            cm = staged_from_gcs(source_ref)
        else:
            from contextlib import nullcontext
            cm = nullcontext(source_ref)

        with cm as local_path:
            raw = load_preprocessed(local_path, preload=True)

            # ── Average reference (before ICA) ────────────────────────────────
            # Must happen here so ICA components reflect the average-referenced
            # scalp field. ICA_N_COMPONENTS=None lets MNE auto-detect rank,
            # which will be n_eeg_channels - 1 after average referencing.
            raw, rank_reduction = apply_average_reference(raw)
            n_eeg = len(mne.pick_types(raw.info, eeg=True))
            # ─────────────────────────────────────────────────────────────────

            ica, fit_info = fit_ica(raw)
            save_ica(ica, entities, deriv_root)

        # Upload Stage 2 result to GCS
        gcs_uri = None
        if UPLOAD_STAGE2_TO_GCS and out_path.exists():
            gcs_uri = upload_stage2_to_gcs(out_path)

        result = {
            **base,
            'status':          'ok',
            'out':             str(out_path),
            'gcs_uri':         gcs_uri,
            'n_eeg_channels':  n_eeg,
            'rank_reduction':  rank_reduction,
            'expected_components': n_eeg - rank_reduction,
            **fit_info,
        }
    except Exception as e:
        result = {**base, 'status': 'fail', 'error': str(e)}
        with open(ERROR_LOG, 'a') as f:
            f.write(f'\n=== {entities} ===\n{traceback.format_exc()}\n')
    finally:
        if raw is not None:
            del raw
        gc.collect()

    if log_file is not None:
        append_log(log_file, result)
    return result


---
# 5. Find input files (from Stage 1)

In [17]:
if STAGE1_SOURCE == 'gcs':
    inputs = discover_preprocessed_files_from_gcs(GCS_STAGE1_PREFIX, tasks=TASKS)
    print(f'Found {len(inputs)} Stage 1 files in GCS')
else:
    inputs = discover_preprocessed_files(STAGE1_DERIV, tasks=TASKS)
    print(f'Found {len(inputs)} Stage 1 files locally')

subjects = set(ent['subject'] for _, ent in inputs)
tasks_found = set(ent['task'] for _, ent in inputs)
print(f'  Subjects: {len(subjects)}')
print(f'  Tasks:    {sorted(tasks_found)}')
for ref, ent in inputs[:3]:
    name = ref.name if hasattr(ref, 'name') else Path(ref).name
    print(f'  {name}')

Found 2171 Stage 1 files locally
  Subjects: 136
  Tasks:    ['FAST', 'IC', 'Restingstate', 'motor']
  sub-10003_task-FAST_run-01_desc-preproc_eeg.fif
  sub-10003_task-IC_run-01_desc-preproc_eeg.fif
  sub-10003_task-Restingstate_run-01_desc-preproc_eeg.fif


---
# 6. Smoke test on one file

**Always run this before launching the full batch.** ICA fits take 1–10 minutes per file 
depending on channel count and convergence. If something is wrong with the config, you 
want to find out at file 1, not file 1500.

In [18]:
test_pair = inputs[0]
ref = test_pair[0]
display_name = ref.name if hasattr(ref, 'name') else Path(ref).name
print(f'🧪 Testing on: {display_name}\n')

test_result = process_one(test_pair, log_file=None)
print(json.dumps(test_result, indent=2, default=str))

🧪 Testing on: sub-10003_task-FAST_run-01_desc-preproc_eeg.fif

{
  "subject": "10003",
  "task": "FAST",
  "run": "01",
  "status": "skipped",
  "out": "/Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-fit/sub-10003/eeg/sub-10003_task-FAST_run-01_desc-preproc_ica.fif"
}


In [19]:
# Verify the output ICA file
test_out = ica_output_path(test_pair[1], DERIV_ROOT)
if test_out.exists():
    ica_check = mne.preprocessing.read_ica(test_out, verbose=False)
    print(f'✅ Output:           {test_out}')
    print(f'   Method:           {ica_check.method}')
    print(f'   N components:     {ica_check.n_components_}')
    print(f'   Iterations:       {getattr(ica_check, "n_iter_", "n/a")}')
    print(f'   File size:        {test_out.stat().st_size / 1e6:.2f} MB')
    print(f'   Channel count:    {len(ica_check.ch_names)}')
    # Uncomment to view component topographies:
    # raw = load_preprocessed(test_pair[0])
    # ica_check.plot_components(inst=raw, picks=range(min(20, ica_check.n_components_)))
else:
    print('❌ No output file produced — check', ERROR_LOG)

✅ Output:           /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-fit/sub-10003/eeg/sub-10003_task-FAST_run-01_desc-preproc_ica.fif
   Method:           picard
   N components:     60
   Iterations:       136
   File size:        0.07 MB
   Channel count:    64


---
# 7. Run the full batch

ICA fitting is **CPU-bound, not I/O-bound** (unlike Stage 1). If you have spare RAM 
(say, 32 GB+), parallelizing across files gives a real speedup. Set `N_JOBS_OUTER` 
accordingly. Watch RAM with `htop` / Activity Monitor on your first parallel run.

In [20]:
if STAGE1_SOURCE == 'gcs':
    inputs = discover_preprocessed_files_from_gcs(GCS_STAGE1_PREFIX, tasks=TASKS)
    print(f'Found {len(inputs)} Stage 1 files in GCS')
else:
    inputs = discover_preprocessed_files(STAGE1_DERIV, tasks=TASKS)
    print(f'Found {len(inputs)} Stage 1 files locally')

Found 2171 Stage 1 files locally


In [ ]:
print(f'🚀 Fitting ICA on {len(inputs)} files')
print(f'   Method:           {ICA_METHOD}  {ICA_FIT_PARAMS}')
print(f'   N components:     {ICA_N_COMPONENTS}  (None = auto from rank)')
print(f'   Decim:            {ICA_DECIM}')
print(f'   Output:           {DERIV_ROOT}')
print(f'   N_JOBS_OUTER:     {N_JOBS_OUTER}')
print(f'   Log:              {LOG_FILE}\n')

if N_JOBS_OUTER == 1:
    for pair in tqdm(inputs, desc='ICA fitting'):
        process_one(pair)
else:
    Parallel(n_jobs=N_JOBS_OUTER, verbose=10)(
        delayed(process_one)(pair) for pair in inputs
    )

print('\n✅ Batch complete.')

🚀 Fitting ICA on 2171 files
   Method:           picard  {'ortho': False, 'extended': True}
   N components:     None  (None = auto from rank)
   Decim:            3
   Output:           /Users/alirezafatemi/asd_eeg_pipeline/derivatives/mne-ica-fit
   N_JOBS_OUTER:     1
   Log:              /Users/alirezafatemi/asd_eeg_pipeline/derivatives/logs/02_ica_fit_log.csv



ICA fitting:   0%|          | 0/2171 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/picard/solver.py:219: UserWarning: Picard did not converge. Final gradient norm : 1.705e-06. Requested tolerance : 1e-07. Consider increasing the number of iterations or the tolerance.
  warnings.warn('Picard did not converge. Final gradient norm : %.4g.'
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/picard/solver.py:219: UserWarning: Picard did not converge. Final gradient norm : 1.512e-05. Requested tolerance : 1e-07. Consider increasing the number of iterations or the tolerance.
  warnings.warn('Picard did not converge. Final gradient norm : %.4g.'
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/picard/solver.py:219: UserWarning: Picard did not converge. Final gradient norm : 3.46e-07. Requested tolerance : 1e-07. Consider increasing the number of iterations or the tolerance.
  warnings.warn('Picard did not converge. Final gradient norm : 


✅ Batch complete.


## 7.1 Upload Stage 2 outputs to GCS

Uploads every `_desc-preproc_ica.fif` file from `DERIV_ROOT` to  
`gs://{GCS_BUCKET}/{GCS_STAGE2_PREFIX}/`.  
Files whose GCS size already matches the local size are **skipped** (cheap idempotent re-runs).  
The GCS client is initialised here on demand, so this cell works even when  
`UPLOAD_STAGE2_TO_GCS = False` was set during the batch run above.


In [24]:
# ── Upload all Stage 2 ICA outputs to GCS ────────────────────────────────────
if not HAS_GCS:
    raise RuntimeError(
        'google-cloud-storage is not installed.\n'
        'Run: pip install google-cloud-storage'
    )

from google.cloud import storage as _gcs_lib
from google.cloud.storage.retry import DEFAULT_RETRY as _DEFAULT_RETRY

_gcs_client  = _gcs_lib.Client()
_gcs_bucket  = _gcs_client.bucket(GCS_BUCKET)
_gcs_retry   = _DEFAULT_RETRY.with_deadline(600).with_delay(
                    initial=2, maximum=30, multiplier=2)
_gcs_timeout = 600
_gcs_chunk   = 16 * 1024 * 1024

local_files = sorted(DERIV_ROOT.rglob('*_desc-preproc_ica.fif'))
print(f'Found {len(local_files)} Stage 2 files to upload'
      f' → gs://{GCS_BUCKET}/{GCS_STAGE2_PREFIX}/')

upload_ok, upload_skip, upload_fail = [], [], []

for local_path in tqdm(local_files, desc='GCS upload'):
    rel       = local_path.relative_to(DERIV_ROOT)
    blob_name = f'{GCS_STAGE2_PREFIX}/{rel.as_posix()}'
    blob      = _gcs_bucket.blob(blob_name, chunk_size=_gcs_chunk)
    try:
        if blob.exists(retry=_gcs_retry, timeout=_gcs_timeout):
            blob.reload(retry=_gcs_retry, timeout=_gcs_timeout)
            if blob.size == local_path.stat().st_size:
                upload_skip.append(blob_name)
                continue
        blob.upload_from_filename(
            str(local_path),
            timeout=_gcs_timeout,
            retry=_gcs_retry,
        )
        upload_ok.append(blob_name)
    except Exception as exc:
        upload_fail.append((str(local_path), str(exc)))

print(f'\n  Uploaded : {len(upload_ok)}')
print(f'  Skipped  : {len(upload_skip)}  (already up-to-date)')
if upload_fail:
    print(f'  Failed   : {len(upload_fail)}')
    for path, err in upload_fail[:10]:
        print(f'    {path}: {err}')
else:
    print('  No failures.')


Found 2168 Stage 2 files to upload → gs://asd-eeg-dataset/derivatives/ds006780/mne-ica-fit/


GCS upload:   0%|          | 0/2168 [00:00<?, ?it/s]


  Uploaded : 2164
  Skipped  : 4  (already up-to-date)
  No failures.


In [ ]:
import pandas as pd
df = pd.read_csv('~/asd_eeg_pipeline/derivatives/logs/02_ica_fit_log.csv')
print(df['status'].avalue_counts())
fails = df[df['status'] == 'fail']
print(fails[['subject','task','run','error']].to_string())

ParserError: Error tokenizing data. C error: Expected 6 fields in line 3, saw 13


---
# 8. Inspect the run

Read the log fresh from disk — we don't keep results in memory.

In [ ]:
log_df = pd.read_csv(LOG_FILE)
print('Status counts:')
print(log_df['status'].value_counts(), '\n')

# Failures
fails = log_df[log_df['status'] == 'fail']
if len(fails):
    print(f'⚠️  {len(fails)} failures — see {ERROR_LOG}')
    print(fails[['subject', 'task', 'run', 'error']].head(20))
else:
    print('🎉 No failures.')

ParserError: Error tokenizing data. C error: Expected 6 fields in line 3, saw 13


In [ ]:
# Convergence + timing diagnostics
ok = log_df[log_df['status'] == 'ok']
if len(ok):
    print('Components fit per file:')
    print(ok['n_components_fit'].describe(), '\n')

    print('Iterations until convergence:')
    print(ok['n_iter'].describe(), '\n')

    print('Fit time per file (seconds):')
    print(ok['fit_seconds'].describe(), '\n')

    # Files that hit max_iter — may not have fully converged
    if 'converged' in ok.columns:
        non_converged = ok[ok['converged'] == False]
        if len(non_converged):
            print(f'⚠️  {len(non_converged)} files did NOT converge — '
                  f'inspect or increase max_iter:')
            print(non_converged[['subject','task','run','n_iter']].head(20))
        else:
            print('✅ All files converged within max_iter.')

NameError: name 'log_df' is not defined

In [ ]:
# ── Average reference + rank audit ───────────────────────────────────────────
# Every file should have expected_components == n_eeg_channels - 1.
# If any file has n_components_fit != expected_components, ICA may have
# used wrong rank — investigate those files.
if len(ok) and 'expected_components' in ok.columns:
    rank_mismatch = ok[ok['n_components_fit'] != ok['expected_components']]
    if len(rank_mismatch):
        print(f'⚠️  {len(rank_mismatch)} files where n_components_fit != expected (n_eeg - 1):')
        print(rank_mismatch[['subject', 'task', 'run',
                               'n_eeg_channels', 'expected_components',
                               'n_components_fit']].head(20))
    else:
        eeg_counts = ok['n_eeg_channels'].unique()
        comp_counts = ok['n_components_fit'].unique()
        print(f'✅ Average reference rank check passed for all {len(ok)} files.')
        print(f'   EEG channel counts seen:   {sorted(eeg_counts)}')
        print(f'   Components fit:            {sorted(comp_counts)}')
        print(f'   (Should be n_eeg - 1 in each case)')


In [ ]:
# Total compute time so far
if len(ok):
    total_hours = ok['fit_seconds'].sum() / 3600
    avg_minutes = ok['fit_seconds'].mean() / 60
    print(f'Total ICA fit time: {total_hours:.1f} hours')
    print(f'Average per file:   {avg_minutes:.1f} minutes')

## Coming next: Stage 3 (ICA label + apply)

1. **Component labeling** — `mne-icalabel` automatically classifies each component as 
   brain / eye / muscle / heart / line noise / channel noise / other. Saves labels as TSV 
   alongside the ICA file.
2. **Component review** (optional) — flag files where >X% of components are bad for manual 
   inspection.
3. **Apply** — load Stage 1 `_eeg.fif` + Stage 2 `_ica.fif` + Stage 3 labels → drop bad 
   components → save `desc-clean_eeg.fif` in a new `mne-ica-apply` derivative.

> ✅ **Note:** Average reference is already applied in this stage (Stage 2), before ICA.  
> Stage 3 does **not** need to re-reference again — the `desc-clean_eeg.fif` output is  
> already on average reference and ready for epoching and PLV computation.

The split into fit / label / apply means you can re-run labeling with different criteria  
(e.g., a stricter threshold) **without** re-fitting ICA. That's the key benefit of staged  
checkpointing.
